# Fase 1 — Representación y espacios de color

Analizamos las **121 imágenes propias** del proyecto (**50 fotografías de productos individuales** en 10 clases y **71 escenas de supermercado** en 10 categorías de estante) evaluando su resolución, canales, memoria en matriz y comportamiento en los espacios de color **RGB, Escala de Grises, Binario y HSV**.

## 1. Configuración y carga de imágenes
Detecta automáticamente `Proyecto Integrador P1/data/propio/imagenes` desde la raíz del espacio de trabajo (`VISION POR COMPUTADOR`), desde `Proyecto Integrador P1` o desde `notebooks/`.

In [1]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

base_candidatos = [Path.cwd(), *Path.cwd().parents]
candidatos = [*(p / 'Proyecto Integrador P1' for p in base_candidatos), *base_candidatos]
PROYECTO = next((p for p in candidatos if (p / 'data' / 'propio' / 'imagenes').is_dir() and (p / 'metadata').is_dir()), None)
if PROYECTO is None:
    raise FileNotFoundError('No encuentro la carpeta Proyecto Integrador P1.')

CARPETA_PROPIA = PROYECTO / 'data' / 'propio' / 'imagenes'
EXTENSIONES = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
rutas = sorted(p for p in CARPETA_PROPIA.rglob('*') if p.is_file() and p.suffix.lower() in EXTENSIONES)
imagenes = {p.name: cv2.imread(str(p), cv2.IMREAD_COLOR) for p in rutas}
imagenes = {nombre: imagen for nombre, imagen in imagenes.items() if imagen is not None}
print('Raíz del proyecto:', PROYECTO)
print(f'Imágenes propias legibles con OpenCV: {len(imagenes)}')

Raíz del proyecto: C:\C_PROJECTS\VISION POR COMPUTADOR\Proyecto Integrador P1
Imágenes propias legibles con OpenCV: 121


### Conclusión

Las **121 imágenes propias** se cargaron y decodificaron correctamente con OpenCV como matrices tridimensionales `BGR` de 8 bits por canal.

## 2. Resolución, canales y tamaño en memoria
OpenCV decodifica estas fotos como matrices BGR de 8 bits por canal (24 bits por píxel). Se compara el peso comprimido en disco (`KB`) frente al consumo real de la matriz en memoria RAM (`MB`).

In [2]:
filas = []
for nombre, bgr in imagenes.items():
    alto, ancho, canales = bgr.shape
    ruta = CARPETA_PROPIA / nombre
    filas.append({
        'imagen': nombre,
        'tipo': 'producto' if nombre.startswith('producto_') else 'escena',
        'ancho_px': ancho,
        'alto_px': alto,
        'canales_cargados': canales,
        'bits_por_canal': bgr.dtype.itemsize * 8,
        'memoria_matriz_MB': round(bgr.nbytes / 1_000_000, 3),
        'archivo_KB': round(ruta.stat().st_size / 1024, 1),
    })
tabla = pd.DataFrame(filas)
display(tabla.groupby('tipo')[['ancho_px', 'alto_px', 'memoria_matriz_MB', 'archivo_KB']].mean().round(2))
display(tabla.head(12))

          ancho_px  alto_px  memoria_matriz_MB  archivo_KB
tipo                                                      
escena     3137.58  3918.42              36.58     2663.81
producto   1296.00  1504.00               5.76      234.52
                              imagen  ...  archivo_KB
0   escena_abarrotes_cereales_01.jpg  ...      3348.8
1   escena_abarrotes_cereales_02.jpg  ...      3108.3
2   escena_abarrotes_cereales_03.jpg  ...      3031.9
3   escena_abarrotes_cereales_04.jpg  ...      3095.8
4   escena_abarrotes_cereales_05.jpg  ...      3200.1
5   escena_abarrotes_cereales_06.jpg  ...      2465.4
6   escena_abarrotes_cereales_07.jpg  ...      3121.8
7   escena_abarrotes_cereales_08.jpg  ...      2733.9
8   escena_abarrotes_cereales_09.jpg  ...      2843.2
9   escena_abarrotes_cereales_10.jpg  ...      3134.7
10  escena_abarrotes_cereales_11.jpg  ...      2675.5
11  escena_abarrotes_cereales_12.jpg  ...      2877.6

[12 rows x 8 columns]


### Conclusión

Cada foto de producto ($1200 \times 1600\text{ px}$) ocupa **$5.76\text{ MB}$** en memoria RAM y cada escena de estante ($3024 \times 4032\text{ px}$) alcanza **$36.58\text{ MB}$** frente a $\sim 3.45\text{ MB}$ en disco. Esta diferencia fundamenta redimensionar las entradas a **$224 \times 224\text{ px}$** para entrenar con bajo costo computacional.

## 3. Galería de fotografías propias
Muestra representativa de productos individuales y escenas de pasillos del supermercado.

In [3]:
muestra_galeria = [
    'producto_cereal_milo_02.jpeg',
    'producto_leche_condensada_03.jpeg',
    'escena_bebidas_01.jpg',
    'escena_snacks_golosinas_01.jpg',
    'escena_frutas_verduras_03.jpg',
    'escena_limpieza_hogar_01.jpg',
]
muestra_galeria = [nombre for nombre in muestra_galeria if nombre in imagenes]
n = len(muestra_galeria)
if n:
    columnas = 3
    filas_grafico = int(np.ceil(n / columnas))
    fig, ejes = plt.subplots(filas_grafico, columnas, figsize=(13, 4 * filas_grafico))
    ejes = np.array(ejes).reshape(-1)
    for ax, nombre in zip(ejes, muestra_galeria):
        bgr = imagenes[nombre]
        ax.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        ax.set_title(nombre)
        ax.axis('off')
    for ax in ejes[n:]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

### Conclusión

La galería visual evidencia las condiciones reales de captura del dataset: diferencias de iluminación entre pasillos, reflejos especulares en envases plásticos y metálicos, fondos complejos y variación de escala entre productos aislados y estanterías.

## 4. Comparación RGB, escala de grises y binaria
La conversión a escala de grises pondera $0.299R + 0.587G + 0.114B$. La binarización con umbral fijo (`127`) muestra por qué la intensidad sola no basta cuando el fondo y las letras del empaque comparten brillo.

In [4]:
nombre_elegido = 'producto_cereal_milo_02.jpeg' if 'producto_cereal_milo_02.jpeg' in imagenes else next(iter(imagenes))
bgr = imagenes[nombre_elegido]
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
gris = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
_, binaria = cv2.threshold(gris, 127, 255, cv2.THRESH_BINARY)
fig, ejes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img, titulo, cmap in zip(
    ejes,
    [rgb, gris, binaria, rgb],
    ['RGB', 'Escala de grises', 'Binaria (umbral 127)', nombre_elegido],
    [None, 'gray', 'gray', None],
):
    ax.imshow(img, cmap=cmap)
    ax.set_title(titulo)
    ax.axis('off')
plt.tight_layout()
plt.show()

### Conclusión

La binarización en escala de grises con umbral fijo (`127`) mezcla el fondo con las letras y zonas claras del propio empaque, demostrando que la intensidad luminosa por sí sola no permite separar el producto del entorno.

## 5. Canales HSV e histogramas
OpenCV representa `H` (tono) en `0–179` y `S`/`V` (saturación y brillo) en `0–255` para imágenes de 8 bits. Desacoplar el tono (`H`) y la saturación (`S`) del brillo (`V`) es clave para aplicar CLAHE solo en `V` sin distorsionar los colores de la marca.

In [5]:
hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
fig, ejes = plt.subplots(1, 3, figsize=(13, 4))
for ax, canal, titulo in zip(ejes, cv2.split(hsv), ['H (tono)', 'S (saturación)', 'V (brillo)']):
    ax.imshow(canal, cmap='gray', vmin=0, vmax=255)
    ax.set_title(titulo)
    ax.axis('off')
plt.tight_layout()
plt.show()

fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
for valores, etiqueta, color in zip(cv2.split(rgb), ['R', 'G', 'B'], ['red', 'green', 'blue']):
    hist, bins = np.histogram(valores, bins=256, range=(0, 256))
    ejes[0].plot(bins[:-1], hist, color=color, label=etiqueta, alpha=0.8)
ejes[0].set(title='Histograma RGB', xlabel='Intensidad', ylabel='Píxeles')
ejes[0].legend()
hist_gris, bins_gris = np.histogram(gris, bins=256, range=(0, 256))
ejes[1].plot(bins_gris[:-1], hist_gris, color='dimgray')
ejes[1].set(title='Histograma en gris', xlabel='Intensidad', ylabel='Píxeles')
plt.tight_layout()
plt.show()

### Conclusión

La descomposición en **HSV** e histogramas confirma que el canal `V` concentra los cambios de iluminación mientras que `H` y `S` preservan el color del envase, justificando aplicar **CLAHE sobre el canal `V`** sin distorsionar el tono del producto.

## 6. Segmentación HSV exploratoria
Demostración de máscara por color en HSV para aislar el tono verde característico del empaque de Milo frente al fondo.

In [6]:
hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
limite_inferior = np.array([30, 50, 35], dtype=np.uint8)
limite_superior = np.array([100, 255, 255], dtype=np.uint8)
mascara_verde = cv2.inRange(hsv, limite_inferior, limite_superior)
segmentada = cv2.bitwise_and(rgb, rgb, mask=mascara_verde)
porcentaje_area = 100 * np.count_nonzero(mascara_verde) / mascara_verde.size
fig, ejes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, titulo, cmap in zip(
    ejes,
    [rgb, mascara_verde, segmentada],
    ['Original', 'Máscara verde HSV', f'Aplicada ({porcentaje_area:.1f}% del área)'],
    [None, 'gray', None],
):
    ax.imshow(img, cmap=cmap)
    ax.set_title(titulo)
    ax.axis('off')
plt.tight_layout()
plt.show()

### Conclusión

La máscara cromática en HSV sobre `producto_cereal_milo_02.jpeg` aísla el color verde del envase ($41.4\%$ del área), mientras que los huecos internos correspondientes al texto y logotipo muestran la necesidad de complementar con operaciones morfológicas de apertura y cierre.

## 7. Conclusión general

El análisis de representación sobre las 121 imágenes demuestra que trabajar en el espacio **HSV** (mejorando el contraste local en `V` con **CLAHE** y utilizando `H`/`S` junto con morfología matemática) proporciona una separación de productos más estable frente a sombras y reflejos que RGB o escala de grises, definiendo el preprocesamiento base para el Spike.